# 🎯 Phase 5: Milestone Solutions

> **Advanced ML & Deep Learning**
>
> This notebook provides theory solutions for the four Real-World Application Scenarios
> described in the Phase Overview. Each scenario requires combining multiple techniques
> from Days 49-60 into a complete system design.

---

## Scenario 1: E-Commerce Recommendation System

**Challenge**: Build an intelligent recommendation system that boosts conversion by 15-25%.

### Architecture Overview

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  User Events  │────▶│   Feature    │────▶│   Model      │
│  (clicks,     │     │   Store      │     │   Ensemble   │
│   purchases)  │     │  (Redis)     │     │              │
└──────────────┘     └──────────────┘     └──────┬───────┘
                                                  │
                     ┌──────────────┐     ┌───────▼───────┐
                     │  A/B Testing  │◀────│   Ranking     │
                     │  Framework    │     │   Service     │
                     └──────────────┘     └──────────────┘
```

### Component Breakdown

**1. Collaborative Filtering (Day 57) — The Core Engine**

- **Approach**: Matrix Factorization (SVD) on user-item interaction matrix
- **Cold Start Problem**: New users with no history → fall back to content-based filtering (item features: category, brand, price range)
- **Implicit Feedback**: Use clicks (weight 1), add-to-cart (weight 3), purchase (weight 5) — not just ratings

```python
# Pseudocode: Alternating Least Squares (ALS) for implicit feedback
# user_factors (U) @ item_factors (V).T ≈ interaction_matrix (R)
#
# For each user u:
#   U[u] = solve(V.T @ C_u @ V + λI, V.T @ C_u @ p_u)
#   where C_u = diagonal confidence matrix, p_u = binary preference vector
```

**Why ALS over SGD?** ALS parallelizes naturally — update all users independently, then all items. Scales to millions of users.

**2. Demand Forecasting (Day 56) — Inventory Planning**

- **Model**: Facebook Prophet for business-friendly seasonality
- **Features**: holidays, promotional events, day-of-week
- **Uncertainty**: Use prediction intervals (80%/95%) for safety stock decisions

**3. NLP Review Analysis (Day 49) — Quality Signals**

- **Sentiment Extraction**: Fine-tuned DistilBERT on product reviews
- **Aspect Mining**: Extract specific praise/complaints ("battery life is great but screen is dim")
- **Review Quality Score**: Feed into recommendation ranking as a signal

**4. MLOps (Day 50) — Production Deployment**

| Concern | Solution |
|---------|----------|
| Latency | Pre-compute top-100 per user, serve from cache |
| Freshness | Retrain weekly, hot-swap model weights |
| Measurement | A/B test CTR, add-to-cart rate, revenue per session |
| Monitoring | Track recommendation diversity, novelty, serendipity |

### Trade-off Analysis

| Approach | Pros | Cons |
|----------|------|------|
| Content-based only | No cold start | Filter bubble, limited discovery |
| Collaborative only | Discovers cross-category patterns | Cold start, popularity bias |
| **Hybrid (recommended)** | Best of both, cold-start handled | More complex, harder to debug |

In [ ]:
# Simplified collaborative filtering demo using NumPy SVD
import numpy as np

# User-Item interaction matrix (5 users × 6 items)
# Values: 0 = no interaction, 1-5 = rating
R = np.array([
    [5, 3, 0, 1, 0, 0],  # User A likes items 1, 2
    [4, 0, 0, 1, 0, 0],  # User B likes item 1
    [1, 1, 0, 5, 0, 0],  # User C likes item 4
    [0, 0, 5, 4, 0, 0],  # User D likes items 3, 4
    [0, 1, 5, 0, 3, 4],  # User E likes items 3, 5, 6
])

# SVD decomposition: R ≈ U @ S @ Vt
U, S, Vt = np.linalg.svd(R, full_matrices=False)

# Keep top-k factors (dimensionality reduction)
k = 3
U_k = U[:, :k]
S_k = np.diag(S[:k])
Vt_k = Vt[:k, :]

# Reconstructed (predicted) matrix
R_pred = U_k @ S_k @ Vt_k

users = ['User A', 'User B', 'User C', 'User D', 'User E']
items = ['Item 1', 'Item 2', 'Item 3', 'Item 4', 'Item 5', 'Item 6']

print("Original Ratings (0 = not rated):")
print(f"{'':>10s}", " ".join(f"{i:>7s}" for i in items))
for u, row in zip(users, R):
    print(f"{u:>10s}", " ".join(f"{v:>7.0f}" for v in row))

print("\nPredicted Scores (including recommendations):")
print(f"{'':>10s}", " ".join(f"{i:>7s}" for i in items))
for u, orig, pred in zip(users, R, R_pred):
    vals = []
    for o, p in zip(orig, pred):
        if o == 0:
            vals.append(f"{p:>6.1f}*")  # New recommendation
        else:
            vals.append(f"{p:>7.1f}")
    print(f"{u:>10s}", " ".join(vals))

print("\n* = Predicted score for items user hasn't rated (recommendation candidates)")

---

## Scenario 2: Healthcare Readmission Prediction

**Challenge**: Predict patient readmission risk to reduce readmissions by 20% and save $10M/year.

### System Design

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  EHR System   │────▶│  Feature     │────▶│  XGBoost     │
│  (500+ vars)  │     │  Selection   │     │  Ensemble    │
│               │     │  (→50 vars)  │     │              │
└──────────────┘     └──────────────┘     └──────┬───────┘
                                                  │
┌──────────────┐     ┌──────────────┐     ┌───────▼───────┐
│  Clinical     │◀────│  Alert       │◀────│  Calibrated   │
│  Dashboard    │     │  System      │     │  Probabilities│
└──────────────┘     └──────────────┘     └──────────────┘
```

### Component Breakdown

**1. Feature Selection (Day 53) — Reduce 500 → 50 Features**

- **Step 1**: Remove near-zero variance features (lab results always normal)
- **Step 2**: Mutual Information ranking — select top features correlated with readmission
- **Step 3**: Recursive Feature Elimination (RFE) with XGBoost
- **Step 4**: Domain expert review — ensure selected features are clinically meaningful

**Key Features** (typical for readmission prediction):

| Category | Features | Why Important |
|----------|----------|---------------|
| Demographics | Age, insurance type | Social determinants |
| Clinical | # prior admissions, LOS, comorbidity index | History predicts future |
| Vitals | Heart rate at discharge, BP trend | Stability indicator |
| Lab | Hemoglobin, creatinine, BNP | Physiological state |
| Discharge | Discharge disposition, follow-up scheduled | Post-discharge plan quality |

**2. XGBoost Ensemble (Day 52) — The Prediction Engine**

```python
# Pseudocode: XGBoost with careful hyperparameter tuning
# Key hyperparameters for healthcare:
#   scale_pos_weight = n_negative / n_positive  (class imbalance)
#   max_depth = 4-6  (prevent overfitting on small cohorts)
#   min_child_weight = 10  (ensure leaf nodes have enough samples)
#   reg_alpha = 0.1, reg_lambda = 1.0  (regularization)
#
# SHAP for interpretability:
#   shap.TreeExplainer(model).shap_values(X_test)
#   → "This patient's readmission risk is high BECAUSE:
#      +12% from 3 prior admissions, +8% from low hemoglobin"
```

**3. Probability Calibration (Day 54) — Clinical Decision Support**

- Raw model outputs are often overconfident — **Platt scaling** or **isotonic regression** to calibrate
- Clinicians need reliable probabilities: "80% chance of readmission" must mean 80% of similar patients actually get readmitted
- **Brier Score** to measure calibration quality

**4. Anomaly Detection (Day 55) — Early Warning**

- **Isolation Forest** on vital signs time series
- Flag unusual readings 6-12 hours before discharge
- Alert: "Patient's BP trajectory is abnormal compared to similar patients"

### Ethical Considerations

- **Fairness**: Check model performance across demographics (race, age, insurance). Disparate impact = bias.
- **Explainability**: SHAP values for every prediction — clinicians must understand WHY
- **Override**: Clinicians can always override the model. It's decision *support*, not decision *making*.

In [ ]:
# Simplified readmission risk scoring demo
import numpy as np

def readmission_risk_score(patient):
    """
    Simplified rule-based readmission risk scorer.

    In production, this would be an XGBoost model with SHAP explanations.
    This demonstrates the logic and feature importance concept.

    Args:
        patient: dict with clinical features

    Returns:
        tuple: (risk_score 0-100, risk_factors list)
    """
    score = 0
    factors = []

    # Prior admissions (strongest predictor)
    prior = patient.get("prior_admissions", 0)
    if prior >= 3:
        score += 25
        factors.append(f"+25: {prior} prior admissions (high risk)")
    elif prior >= 1:
        score += 10
        factors.append(f"+10: {prior} prior admission(s)")

    # Length of stay
    los = patient.get("length_of_stay", 0)
    if los > 7:
        score += 15
        factors.append(f"+15: Long stay ({los} days)")

    # Comorbidity index
    cci = patient.get("charlson_index", 0)
    if cci >= 5:
        score += 20
        factors.append(f"+20: High comorbidity (CCI={cci})")
    elif cci >= 2:
        score += 10
        factors.append(f"+10: Moderate comorbidity (CCI={cci})")

    # Lab values
    hgb = patient.get("hemoglobin", 14)
    if hgb < 10:
        score += 15
        factors.append(f"+15: Low hemoglobin ({hgb} g/dL)")

    # Follow-up scheduled
    if not patient.get("followup_scheduled", True):
        score += 10
        factors.append("+10: No follow-up appointment scheduled")

    # Age factor
    age = patient.get("age", 50)
    if age > 75:
        score += 10
        factors.append(f"+10: Advanced age ({age})")

    return min(score, 100), factors


# Test with sample patients
patients = [
    {"name": "Patient A", "age": 45, "prior_admissions": 0, "length_of_stay": 3,
     "charlson_index": 1, "hemoglobin": 14, "followup_scheduled": True},
    {"name": "Patient B", "age": 72, "prior_admissions": 2, "length_of_stay": 8,
     "charlson_index": 4, "hemoglobin": 11, "followup_scheduled": True},
    {"name": "Patient C", "age": 80, "prior_admissions": 4, "length_of_stay": 12,
     "charlson_index": 6, "hemoglobin": 8.5, "followup_scheduled": False},
]

print("=" * 55)
print("READMISSION RISK ASSESSMENT")
print("=" * 55)

for p in patients:
    score, factors = readmission_risk_score(p)
    level = "🟢 LOW" if score < 25 else "🟡 MODERATE" if score < 50 else "🔴 HIGH"
    print(f"\n{p['name']} (age {p['age']}): {level} — Score: {score}/100")
    for f in factors:
        print(f"  {f}")

---

## Scenario 3: Social Media Content Moderation

**Challenge**: Automate 90% of content moderation, achieving <100ms latency at 1M requests/min.

### Multi-Stage Pipeline

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ Content   │──▶│ Fast     │──▶│ Deep     │──▶│ Human    │
│ Submitted │   │ Filter   │   │ Analysis │   │ Review   │
│           │   │ (<10ms)  │   │ (<100ms) │   │ (edge)   │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
    100%           Blocks        Analyzes       Reviews
                   5% spam       90% auto       5% edge cases
```

### Stage 1: Fast Filter (<10ms) — Keyword + Regex

- Bloom filter with known bad terms (O(1) lookup, 0.1% false positive rate)
- Regex patterns for known attack vectors (Unicode homoglyphs, leetspeak)
- **Blocks**: Clear-cut violations (known slurs, spam URLs)

### Stage 2: Transformer Model (<100ms) — Fine-tuned BERT

**Model Architecture** (Day 58):
- Base: DistilBERT (6 layers vs BERT's 12 = 60% faster, 97% accuracy)
- Fine-tuned on platform-specific labeled data
- Multi-label output: [toxic, severe_toxic, obscene, threat, insult, identity_hate]

```python
# Pseudocode: Fine-tuning pipeline
# 1. tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
# 2. model = AutoModelForSequenceClassification.from_pretrained(
#        'distilbert-base-uncased', num_labels=6, problem_type='multi_label_classification'
#    )
# 3. Train with focal loss (addresses class imbalance for rare categories)
# 4. Export with ONNX for production serving
```

**Latency Optimization**:
- ONNX Runtime + TensorRT for GPU inference
- Dynamic batching: accumulate requests for 5ms, batch-process
- Model distillation: train smaller model on larger model's outputs

### Stage 3: Graph Analysis — Coordinated Campaigns (Day 60)

- **GNN** on user interaction graph: who likes/shares/replies to same content
- Detect clusters of accounts acting in coordination (bot networks)
- Features: account age, posting frequency, connection patterns

### Stage 4: Data Augmentation (Day 59)

- **Problem**: Hate speech is rare (1-5% of content)
- **Solution**: Use back-translation augmentation + GPT-generated synthetic examples
- **Careful**: Generated examples must be reviewed — don't train on model hallucinations

### Adversarial Robustness

| Attack | Mitigation |
|--------|------------|
| Character substitution (h8te → hate) | Character-level model + normalization |
| Unicode homoglyphs (а → a) | Unicode normalization (NFKD) |
| Context manipulation ("I would NEVER say...") | Sequence classification, not keyword |
| Adversarial images with text | OCR + text pipeline on extracted text |

In [ ]:
# Simplified content moderation pipeline demo
import re

# Stage 1: Fast keyword filter
BLOCKED_PATTERNS = [
    r'\b(spam|scam|click here|free money)\b',
    r'https?://(?:bit\.ly|tinyurl)',  # Suspicious shortened URLs
]

def fast_filter(text):
    """Stage 1: <10ms keyword + regex filter."""
    lower = text.lower()
    for pattern in BLOCKED_PATTERNS:
        if re.search(pattern, lower):
            return "BLOCKED", "Matched known spam pattern"
    return "PASS", None


# Stage 2: Simplified toxicity scorer (simulates transformer output)
TOXIC_SIGNALS = {
    'hate': ['hate', 'disgusting', 'terrible people'],
    'threat': ['kill', 'destroy', 'attack you'],
    'insult': ['stupid', 'idiot', 'moron', 'loser'],
    'obscene': ['damn', 'hell'],
}

def toxicity_score(text):
    """Stage 2: Simulated transformer multi-label output."""
    lower = text.lower()
    scores = {}
    for category, keywords in TOXIC_SIGNALS.items():
        matched = sum(1 for kw in keywords if kw in lower)
        scores[category] = min(matched * 0.35, 0.95)
    return scores


def moderate(text, threshold=0.5):
    """
    Full moderation pipeline.

    Returns:
        tuple: (action, reason, details)
    """
    # Stage 1
    status, reason = fast_filter(text)
    if status == "BLOCKED":
        return "🚫 BLOCKED", reason, {}

    # Stage 2
    scores = toxicity_score(text)
    max_cat = max(scores, key=scores.get)
    max_score = scores[max_cat]

    if max_score >= threshold:
        return "⚠️ FLAGGED", f"{max_cat} ({max_score:.0%})", scores
    elif max_score >= threshold * 0.5:
        return "👀 REVIEW", f"Borderline {max_cat}", scores
    else:
        return "✅ APPROVED", "Clean content", scores


# Test
test_content = [
    "Great article, thanks for sharing!",
    "Check out this deal at https://bit.ly/free123",
    "What a stupid idea, the author is an idiot",
    "I respectfully disagree with the premise",
    "This is terrible people need to hate this moron",
]

print("=" * 55)
print("CONTENT MODERATION PIPELINE")
print("=" * 55)

for content in test_content:
    action, reason, scores = moderate(content)
    print(f"\n{action}: \"{content[:50]}{'...' if len(content) > 50 else ''}\"")
    print(f"  Reason: {reason}")
    if scores:
        print(f"  Scores: {', '.join(f'{k}={v:.0%}' for k, v in scores.items())}")

---

## Scenario 4: Financial Fraud Detection System

**Challenge**: Catch 40% more fraud while reducing false positives by 60%.

### Multi-Layer Architecture

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│Transaction│──▶│ Rules    │──▶│ ML       │──▶│ Graph    │
│ Stream    │   │ Engine   │   │ Anomaly  │   │ Network  │
│           │   │ (<1ms)   │   │ (<10ms)  │   │ Analysis │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
                  Known          Unknown        Ring / Org
                  patterns       anomalies      fraud
```

### Layer 1: Rules Engine (<1ms) — Known Patterns

- Card-not-present + amount > $1000 + first-time merchant → FLAG
- Multiple transactions in different countries within 1 hour → BLOCK
- Transaction at 3 AM + amount > 2× average → FLAG
- **Fast**: Simple if-then rules, sub-millisecond

### Layer 2: Anomaly Detection (Day 55) — Unknown Patterns

**Isolation Forest**: Randomly partition the feature space. Fraudulent transactions (outliers) require fewer splits to isolate.

**Feature Engineering**:

| Feature | Description | Why Important |
|---------|-------------|---------------|
| `velocity_1h` | Transactions in last hour | Normal: 0-2, Fraud: 5-20 |
| `distance_from_avg` | km from user's usual location | Normal: <50km, Fraud: >500km |
| `amount_zscore` | How unusual is this amount? | >3σ = very unusual |
| `merchant_risk` | Historical fraud rate for merchant | Some merchants are riskier |
| `hour_of_day` | When (encoded cyclically) | Fraud peaks 1-5 AM |
| `device_fingerprint_new` | Is this a new device? | New device + high amount = risky |

### Layer 3: Graph Neural Network (Day 60) — Ring Detection

**Key Insight**: Individual fraud is detectable by anomaly detection. **Organized fraud rings** are only detectable by analyzing the network.

**Graph Construction**:
- **Nodes**: Accounts, merchants, devices, IP addresses
- **Edges**: Transactions, shared devices, shared IPs, money transfers
- **GNN**: 2-layer Graph Attention Network (GAT) with attention on edge types

**What the GNN detects**:
- Account A sends money to B, B to C, C back to A (circular flow)
- 10 accounts created same day, same IP, all transacting with same merchant
- Account shares device fingerprint with known fraudulent account

### Layer 4: Time Series Patterns (Day 56)

- Track spending velocity: sudden spike = potential account takeover
- Seasonal patterns: legitimate spending changes predictably (holidays, paydays)
- **LSTM** on transaction sequence: learn normal user behavior → flag deviations

### Ensemble Decision

```python
# Final scoring: weighted combination of all layers
# final_score = w1*rules + w2*anomaly + w3*graph + w4*timeseries
#
# Decision:
#   score > 0.9  → AUTO BLOCK (0.1% of txns)
#   score > 0.6  → MANUAL REVIEW (1% of txns)
#   score > 0.3  → ADDITIONAL AUTH (SMS verification)
#   score < 0.3  → APPROVE
```

In [ ]:
# Simplified fraud detection pipeline demo
import numpy as np

def fraud_risk_score(transaction):
    """
    Multi-layer fraud scoring (simplified).

    In production, each layer would be a separate ML model.
    This demonstrates the scoring logic and feature usage.

    Args:
        transaction: dict with transaction features

    Returns:
        tuple: (score 0-1, decision, explanations)
    """
    score = 0
    explanations = []

    # Layer 1: Rules
    if transaction["amount"] > 2000 and transaction["hour"] < 5:
        score += 0.3
        explanations.append("Rule: High amount at unusual hour")
    if transaction["is_international"] and not transaction.get("travel_history", False):
        score += 0.2
        explanations.append("Rule: International without travel history")

    # Layer 2: Anomaly signals
    if transaction["velocity_1h"] > 5:
        score += 0.25
        explanations.append(f"Anomaly: {transaction['velocity_1h']} txns in last hour")
    if transaction["distance_km"] > 500:
        score += 0.2
        explanations.append(f"Anomaly: {transaction['distance_km']}km from usual location")

    # Layer 3: Device risk
    if transaction.get("new_device", False):
        score += 0.15
        explanations.append("Device: First time from this device")

    score = min(score, 1.0)

    # Decision
    if score > 0.7:
        decision = "🚫 BLOCK"
    elif score > 0.4:
        decision = "⚠️ REVIEW"
    elif score > 0.2:
        decision = "🔐 VERIFY"
    else:
        decision = "✅ APPROVE"

    return score, decision, explanations


# Test transactions
transactions = [
    {"id": "TXN001", "amount": 45.99, "hour": 14, "is_international": False,
     "velocity_1h": 1, "distance_km": 5, "new_device": False},
    {"id": "TXN002", "amount": 3500, "hour": 3, "is_international": True,
     "velocity_1h": 8, "distance_km": 4000, "new_device": True},
    {"id": "TXN003", "amount": 150, "hour": 10, "is_international": False,
     "velocity_1h": 3, "distance_km": 20, "new_device": True},
    {"id": "TXN004", "amount": 890, "hour": 22, "is_international": True,
     "velocity_1h": 6, "distance_km": 800, "new_device": False, "travel_history": True},
]

print("=" * 55)
print("FRAUD DETECTION PIPELINE")
print("=" * 55)

for txn in transactions:
    score, decision, reasons = fraud_risk_score(txn)
    print(f"\n{txn['id']}: ${txn['amount']:,.2f} at {txn['hour']:02d}:00")
    print(f"  {decision} (score: {score:.0%})")
    for r in reasons:
        print(f"  → {r}")

---

## 🎓 Summary

This notebook provided theory solutions and working demos for the four Phase 5 Real-World Application Scenarios:

1. **E-Commerce Recommendations**: SVD-based collaborative filtering + content-based hybrid, with demand forecasting and NLP review analysis
2. **Healthcare Readmission**: Feature selection (500→50), XGBoost with SHAP, calibrated probabilities, anomaly alerts
3. **Content Moderation**: 3-stage pipeline (regex → DistilBERT → GNN), adversarial robustness, <100ms latency at scale
4. **Fraud Detection**: 4-layer system (rules → Isolation Forest → GNN ring detection → LSTM sequences)

### Cross-Cutting Themes

- **Multi-model ensembles**: No single technique solves real-world problems — combine layers
- **Latency vs accuracy trade-off**: Fast filters first, deep analysis second
- **Interpretability**: SHAP, feature importance, and clear explanations are non-negotiable
- **MLOps**: Every system needs monitoring, A/B testing, and retraining pipelines